#agent structure

1. **Database Access**
   1.1. The agent connects to an Oracle database using **SQLAlchemy**.
   1.2. Only **SELECT queries** are executed to ensure safety and prevent accidental modifications.

2. **Table Selection**
   2.1. An **orchestrator function** decides which table the agent should use for a user’s question.
   2.2. The function sends the question to the **LLM**, which selects the most appropriate table based on context.

3. **Table Context with Pydantic**
   3.1. Tables are represented as attributes in a **Pydantic class**, each corresponding to a database table.
   3.2. The `Field(description=...)` contains information about the table: its purpose, contained data, and what can be returned.
   3.3. These descriptions provide **context for the LLM**, helping it choose the correct table and reduce errors.

4. **Exploring the Selected Table**
   4.1. Once a table is chosen, the agent can perform a **table description (`get_desc`)** to inspect columns, data types, and other metadata.
   4.2. This helps the agent construct queries correctly and understand the table structure.

5. **Data Retrieval (Query Execution)**
   5.1. The agent builds **filtered and limited queries**, always prioritizing:
       - Relevant filters extracted from the user’s question to reduce returned data.
       - Row limits (`LIMIT` or Oracle equivalent) to avoid overloading the LLM’s context.
   5.2. This prevents excessive data consumption and keeps query results manageable.

6. **Execution Pipeline**
   6.1. User question → Table orchestrator chooses table → `get_desc` if necessary → `get_select` with filters and limits → LLM generates final response.
   6.2. The design is modular, safe, and scalable, making it easy to maintain and add new tables.

7. **Best Practices**
   7.1. Never return all rows from large tables.
   7.2. Keep table descriptions up-to-date for accurate LLM context.
   7.3. Validate filters before executing queries.
   7.4. Limit queries to **SELECT only** for security.

In [136]:
#imports
from pydantic import BaseModel,Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict
from dotenv import load_dotenv
from typing import Annotated, TypedDict, List
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from langgraph.graph import add_messages
from langchain.tools import tool

In [137]:
#agent state

class State(TypedDict):
    message: Annotated[list,add_messages]
    

In [138]:
#env config / variables
class Environment(BaseSettings):
    table_1: str = Field(alias="TABLE1")
    table_2: str = Field(alias="TABLE2")
    api_deepseek: SecretStr = Field(description="DeepSeek Api Key")
    model_config = SettingsConfigDict(env_file=".env_database_agent")

env = Environment()
    

In [139]:
#class model Tables
class Tables(BaseSettings):
    
    table1: str = Field(
        default=env.table_1,
        description="Table that stores alert records across multiple domains, including infrastructure, code, and application errors."
    )
    
    table2: str = Field(
        default=env.table_2,
        description="Table containing customer support tickets and service interactions, including requests, issues reported by clients, and records of assistance provided."
    )

In [140]:
#model choice table
model_llm = ChatDeepSeek(model="deepseek-chat", temperature=0.7)
def init_model_llm(input, tables_list):
    prompt = f"""You are an assistant that helps decide **which database table to use** to answer a user’s question.

    You are given:
    1. The user’s question: {input}
    2. A list of available tables, each with:
    - The table name
    - A description of what the table contains
    
    --List of available tables {tables_list}

    Available tables:
    - "Alerts Table": Stores error alerts, logs, and technical failures from infrastructure or applications.
    - "Support Tickets Table": Stores customer support tickets, service requests, and records of assistance provided.

    **Task:** Choose **only one table** that best matches the user’s question.  
    **Expected output:** JSON in the format:
    {{
    "table_name": "<name of the chosen table>"
    }}

    **Examples:**  
    Question: "What application errors occurred yesterday?"  
    Output: {{"table_name": "Alerts Table"}}

    Question: "Which customer support tickets were resolved today?"  
    Output: {{"table_name": "Support Tickets Table"}}

    **Important:**  
    - Return **only the JSON**, nothing else.  
    - Choose only one table."""
    
    response = model_llm.invoke(input=prompt).content

    return response


In [141]:
#decide table - tool
#@tool
def decide_table(user_question:str):
    """A tool responsible for selecting the most appropriate table to construct SQL queries 
    based on the user’s input question.

    This tool uses table descriptions (from a Pydantic model) as context, 
    allowing the LLM to determine which table contains the relevant data. 
    It outputs only the table name, which will then be used for further actions 
    such as describing the table columns or executing filtered SELECT queries.
"""
    tables = Tables()
    tables_return = []
    
    for table_name in tables:
        tables_return.append({"table_name":table_name[1], "table_description":Tables.model_fields[f"{table_name[0]}"].description})
        
    response = init_model_llm(input=user_question, tables_list=tables_return)
    
    print(response)
    

In [142]:
decide_table("qual a quantidade total de chamados existentes")

{"table_name": "BEG_CHAMADOS"}
